# LFM2.5-2.6B — Modal Notebook, GPU in the kernel

Serve LFM2.5-2.6B with vLLM **inside this notebook's own GPU kernel** and drive
the NextSearch harness against `localhost` — no separate Modal app, everything
in one place.

**Kernel setup (left panel):**
- **GPU: L4** (the 2.6B model fits easily; A10/A100 also fine).
- **RAM ≳ 16 GB, CPU ≳ 4** — 256 MiB / 0.125 cores cannot load a model.
- Attach your **`huggingface-secret`** (its keys — `PARALLEL_API_KEY`,
  `GEMINI_API_KEY`, `HF_TOKEN` — arrive as environment variables).
- Optional: attach a **Volume** (mounts at `/mnt/…`) for run artifacts that
  survive kernel restarts.

Trade-off vs. the separate Modal app (`deploy/modal_lfm_server.py`): this is
simpler and has no cold-start URL, but the **GPU bills the whole time the kernel
is running** (your idle timeout stops it). Run cells **one at a time**, not
"Run all" — serving and rollouts are long-running.

## 1 · Install the repo + vLLM

In [ ]:
import os
if not os.path.isdir("NextSearch"):
    !git clone https://github.com/neumbilly/NextSearch.git
%cd NextSearch
# Hard-sync to the pushed commit so re-running this cell actually updates the
# code (a plain fetch+checkout won't fast-forward when already on the branch).
# After it changes, RESTART THE KERNEL so Python reloads the modules.
!git fetch origin cursor/lfm2.5-2.6b-stage1-3be6 --quiet && git checkout cursor/lfm2.5-2.6b-stage1-3be6 --quiet && git reset --hard origin/cursor/lfm2.5-2.6b-stage1-3be6 --quiet
!pip install --quiet -e ".[experiment]"
# LFM2.5 support + the lfm2 tool parser ship in vLLM >= 0.23.0.
!pip install --quiet "vllm>=0.23.0"
# If transformers later fails importing torchaudio with a CUDA-version
# mismatch, uncomment the next line (a text agent needs no audio):
# !pip uninstall --quiet -y torchaudio
print("installed from", os.getcwd())

## 2 · Environment: secrets, endpoint (localhost), persistent run root

In [ ]:
import os
# Secrets are already env vars via the attached Modal secret.
print("keys present:", {k: bool(os.environ.get(k))
                        for k in ("PARALLEL_API_KEY", "GEMINI_API_KEY", "HF_TOKEN")})
if os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ["HF_TOKEN"])

# We serve locally in this kernel, so the endpoint is localhost.
BASE_URL = "http://localhost:8000/v1"
os.environ["NEXTSEARCH_BASE_URL"] = BASE_URL

# Persist artifacts on an attached Volume if present, else kernel-local.
import glob
vol = sorted(glob.glob("/mnt/*"))
os.environ["NEXTSEARCH_HOME"] = (vol[0] + "/nextsearch-lfm") if vol else "nextsearch-runs"
print("endpoint:", BASE_URL, "| NEXTSEARCH_HOME:", os.environ["NEXTSEARCH_HOME"])

CFG = dict(model="lfm2.5-2.6b", context_window=32768, harness_cap=28000,
           gpu_util=0.90, max_num_seqs=32, task_date="2026-07-31",
           gpu_label="L4", gpu_hourly_usd=0.80, dev_n=20)

## 3 · Start vLLM in this kernel ⚠️ needs the GPU

`subprocess.Popen` (handle kept + `atexit` cleanup, not `nohup`); logs stream to
a file; a bounded readiness loop polls `/v1/models` and prints the log on
failure. First start downloads the weights, so give it a few minutes.

In [ ]:
import atexit, subprocess, sys, time, urllib.request
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU on this kernel. Set GPU=L4 (and RAM >= 16GB) "
                       "in the kernel panel, restart, and re-run from cell 1.")

VLLM_LOG = "/tmp/vllm.log"
_server = {"proc": None}

def start_vllm():
    if _server["proc"] and _server["proc"].poll() is None:
        print("vLLM already running (pid", _server["proc"].pid, ")"); return
    cmd = ["vllm", "serve", "LiquidAI/LFM2.5-2.6B",
           "--served-model-name", "LiquidAI/LFM2.5-2.6B",
           "--host", "0.0.0.0", "--port", "8000",
           "--max-model-len", str(CFG["context_window"]),
           "--max-num-seqs", str(CFG["max_num_seqs"]),
           "--gpu-memory-utilization", str(CFG["gpu_util"]),
           "--enable-auto-tool-choice", "--tool-call-parser", "lfm2"]
    print("launching:", " ".join(cmd))
    _server["proc"] = subprocess.Popen(cmd, stdout=open(VLLM_LOG, "w"),
                                       stderr=subprocess.STDOUT)
    atexit.register(stop_vllm)

def stop_vllm():
    p = _server.get("proc")
    if p and p.poll() is None:
        p.terminate()
        try: p.wait(timeout=30)
        except subprocess.TimeoutExpired: p.kill()
    _server["proc"] = None

def wait_ready(timeout_s=900, interval_s=5):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if _server["proc"] and _server["proc"].poll() is not None:
            print("vLLM exited early — last 60 log lines:")
            print("".join(open(VLLM_LOG).readlines()[-60:]))
            raise RuntimeError("vLLM process exited during startup")
        try:
            with urllib.request.urlopen(BASE_URL + "/models", timeout=5) as r:
                if r.status == 200: print("vLLM ready"); return True
        except Exception: pass
        time.sleep(interval_s)
    print("".join(open(VLLM_LOG).readlines()[-60:]))
    raise TimeoutError(f"vLLM not ready after {timeout_s}s")

start_vllm(); wait_ready()

## 4 · Compatibility gate (synthetic tool — no search credits)

In [ ]:
import json
from nextsearch.compat import probe
from nextsearch.models import get_client, get_model
m = get_model(CFG["model"], base_url=BASE_URL)
result = await probe(get_client(m), m.model_id, m.sampling)
print(json.dumps(result, indent=2))
assert result["passed"], "compatibility gate FAILED — see checks above"

## 5 · Prepare SEAL-0 + one ungraded smoke rollout, full trace saved

In [ ]:
from pathlib import Path
from nextsearch import benchmarks, harnesses
from nextsearch.compat import _run_coro
from nextsearch.experiment.trace import print_trace, save_as_rollouts, save_trace
from nextsearch.harness import run_episode

benchmarks.get("seal0").prepare()
h = harnesses.get("solo")
row = benchmarks.get("seal0").load_rows(1)[0]
ro = _run_coro(run_episode(get_client(m), m, row, h.tools(), max_turns=10,
    max_context=CFG["harness_cap"], system_suffix=h.system_suffix(CFG["task_date"]),
    bench="seal0"))
RUN_DIR = Path(os.environ["NEXTSEARCH_HOME"]) / "smoke"
save_trace(ro, RUN_DIR / f"{row.id}.trace.json")
save_as_rollouts(ro, RUN_DIR / CFG["model"] / "rollouts.jsonl")
print_trace(ro)

## 6 · Development rollout (N tasks, persisted)

In [ ]:
!nextsearch-eval rollout --benches seal0:{CFG['dev_n']} --models {CFG['model']} --base-url {BASE_URL} --date {CFG['task_date']}

## 7 · Live telemetry + training curves (inline, no tracker)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from nextsearch.experiment.viewer import render, render_curves
render(os.environ["NEXTSEARCH_HOME"], gpu=CFG["gpu_label"], gpu_hourly_usd=CFG["gpu_hourly_usd"]); plt.show()
render_curves(os.environ["NEXTSEARCH_HOME"]); plt.show()

## 8 · (Optional) grade with Gemini + report

In [ ]:
import subprocess, sys
from pathlib import Path
runs = Path(os.environ["NEXTSEARCH_HOME"]) / "runs"
eid = sorted(p.name for p in runs.iterdir() if (p / "manifest.json").exists())[-1]
for stage in (["grade","--eval-id",eid,"--judge","gemini-3.6-flash"], ["report","--eval-id",eid]):
    print(subprocess.run([sys.executable,"-m","nextsearch.cli",*stage],
                         capture_output=True, text=True, env=os.environ).stdout[-3000:])

## 9 · Stop vLLM (frees the GPU; the kernel keeps running)

In [ ]:
stop_vllm(); print("vLLM stopped")